# `google_translator_local.py` — Vietnamese → English Translation

## Purpose
This script translates the **raw Vietnamese cosmetics review dataset** into English using Google Translate (via the `deep-translator` library).

## Pipeline
```
Input : data/raw/full_data_vn.csv   (original Vietnamese reviews)
Output: data/raw/full_data_en.csv   (machine-translated English reviews)
```

## Key design decisions
- Translation is done **row-by-row** with a progress bar (tqdm) so you can monitor how many rows are done.
- If the Google Translate API hits a rate limit or throws an error, the **original text is kept as fallback** — so the script never crashes or loses data.
- Supports any text column name (`data`, `review`, or `text`) by auto-detecting which one is present.
- Windows-specific Unicode fix applied so progress bars display correctly in PowerShell.

In [7]:
print("⏳ Starting: Loading dependencies and libraries...")
import time
_start_time = time.time()
# -*- coding: utf-8 -*-
import pandas as pd
from deep_translator import GoogleTranslator  # Wrapper around Google Translate v2 API
from tqdm import tqdm                         # Progress bar for row-by-row translation
import os
import sys

# ── Windows Unicode fix ────────────────────────────────────────────────────────
# PowerShell's default encoding can't handle Vietnamese characters.
# Wrapping stdout forces UTF-8 output so progress logs don't crash.
if sys.platform == 'win32' and hasattr(sys.stdout, 'buffer'):
    import io
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Loading dependencies and libraries.")


⏳ Starting: Loading dependencies and libraries...
🕒 Done in 0.00s
✅ Completed: Loading dependencies and libraries.


## 1. File Path Configuration
Edit these two paths if your data lives somewhere else.

In [8]:
print("⏳ Starting: Loading dataset...")
import time
_start_time = time.time()
# Run from the ml-research root directory
ML_RESEARCH = os.path.dirname(os.path.abspath(''))
os.chdir(ML_RESEARCH)
input_file  = os.path.join(ML_RESEARCH, 'data', 'raw', 'full_data_vn.csv')  # Raw Vietnamese reviews
output_file = os.path.join(ML_RESEARCH, 'data', 'raw', 'full_data_en.csv')  # Translated English output

# Guard: fail early with a helpful message if the file is missing
if not os.path.exists(input_file):
    print(f"ERROR: Input file not found at: {input_file}")
    print("Please update 'input_file' with the correct path.")
    raise SystemExit(1)

print(f"Loading data from: {input_file}")
print("📖 Reading the dataset from CSV...")
df = pd.read_csv(input_file)
print(f"Loaded {len(df)} rows")
df.head()  # Quick sanity check — see what the raw data looks like
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Loading dataset.")


⏳ Starting: Loading dataset...
Loading data from: c:\Users\lucif\Desktop\Clearview\ml-research\data\raw\full_data_vn.csv
📖 Reading the dataset from CSV...
Loaded 16227 rows
🕒 Done in 0.06s
✅ Completed: Loading dataset.


## 2. Initialize the Translator
`source='auto'` lets Google detect the language automatically. If the reviews were from a different country, this handles multilingual batches without code changes.

In [9]:
print("⏳ Starting: Loading dependencies and libraries...")
import time
_start_time = time.time()
# source='auto' → auto-detect (works even if some rows are already English)
# target='en'   → translate to English
print("🌐 Connecting to Google Translate...")
translator = GoogleTranslator(source="auto", target="en")
print("⚙️ Running a complex transformation on all rows... this might take few mins")
tqdm.pandas()  # Enables df.progress_apply() — shows a progress bar during apply
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Loading dependencies and libraries.")


⏳ Starting: Loading dependencies and libraries...
🌐 Connecting to Google Translate...
⚙️ Running a complex transformation on all rows... this might take few mins
🕒 Done in 0.00s
✅ Completed: Loading dependencies and libraries.


## 3. Safe Translation Function
We wrap the API call in a try/except so a single failed request doesn't abort the whole job.

In [10]:
print("⏳ Starting: Defining function safe_translate...")
import time
_start_time = time.time()
print("🔄 Beginning the translation process... patience is a virtue!")
def safe_translate(text):
    """Safely translate text, handling edge cases and API errors."""
    # Skip null, non-string, or blank entries — return as-is
    if pd.isna(text) or not isinstance(text, str) or not text.strip():
        return text
    try:
        return translator.translate(text)  # Call Google Translate API
    except Exception as e:
        print(f"\nTranslation error: {e}")
        return text  # Fallback: keep the original Vietnamese text on failure
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining function safe_translate.")


⏳ Starting: Defining function safe_translate...
🔄 Beginning the translation process... patience is a virtue!
🕒 Done in 0.00s
✅ Completed: Defining function safe_translate.


## 4. Translate the Review Text Column
Auto-detects the text column name — handles multiple possible column names from different raw datasets.

In [ ]:
print("⏳ Starting: Loading dependencies and libraries...")
import time
_start_time = time.time()
# Detect which column holds the review text and translate it
if 'data' in df.columns:
    print("Translating 'data' column...")
    print("⚙️ Running a complex transformation on all rows... this might take a minute.")
    df["data_en"] = df["data"].progress_apply(safe_translate)  # Adds translated column
elif 'review' in df.columns:
    print("Translating 'review' column...")
    df["review_en"] = df["review"].progress_apply(safe_translate)
elif 'text' in df.columns:
    print("Translating 'text' column...")
    df["text_en"] = df["text"].progress_apply(safe_translate)
else:
    print(f"ERROR: No text column found. Available: {list(df.columns)}")
    raise SystemExit(1)
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Loading dependencies and libraries.")


⏳ Starting: Loading dependencies and libraries...
Translating 'data' column...
⚙️ Running a complex transformation on all rows... this might take a minute.


  0%|          | 26/16227 [00:15<2:39:38,  1.69it/s]

## 5. Save the Translated Dataset

In [ ]:
print("⏳ Starting: Loading dependencies and libraries...")
import time
_start_time = time.time()
print(f"\nSaving to: {output_file}")
print("💾 Saving progress to a new CSV file...")
df.to_csv(output_file, index=False)  # Write translated DataFrame back to CSV
print(f"Successfully saved → {output_file}")
print(f"Total rows translated: {len(df)}")
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Loading dependencies and libraries.")


⏳ Starting: Loading dependencies and libraries...

Saving to: c:\Users\lucif\Desktop\Clearview\ml-research\data\raw\full_data_en.csv
💾 Saving progress to a new CSV file...
Successfully saved → c:\Users\lucif\Desktop\Clearview\ml-research\data\raw\full_data_en.csv
Total rows translated: 16227
🕒 Done in 0.07s
✅ Completed: Loading dependencies and libraries.
